<a href="https://colab.research.google.com/github/racoope70/daytrading-with-ml/blob/main/train_extended_ML_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install stable-baselines3[extra] gymnasium gym-anytrading yfinance xgboost joblib tensorflow

In [2]:
!pip uninstall -y dask cudf-cu12 cuml-cu12 rapids-dask-dependency pylibraft-cu12 pylibcudf-cu12 numba stable-baselines3 gymnasium gym-anytrading

In [3]:
!pip install --upgrade --force-reinstall \
    dask==2024.11.2 \
    rapids-dask-dependency==24.12.0 \
    cudf-cu12==24.12.0 \
    cuml-cu12==24.12.0 \
    pylibraft-cu12==24.12.0 \
    pylibcudf-cu12==24.12.0 \
    numba==0.61.0 \
    stable-baselines3[extra] \
    gymnasium==0.29.1 \
    gym-anytrading==2.0.0


In [1]:
import cudf, cuml, dask, stable_baselines3, gymnasium
import numba, pandas, numpy, scipy

print("cuDF Version:", cudf.__version__)
print("cuML Version:", cuml.__version__)
print("Dask Version:", dask.__version__)
print("Stable Baselines3 Installed:", stable_baselines3.__version__)
print("Gymnasium Version:", gymnasium.__version__)
print("NumPy Version:", numpy.__version__)
print("SciPy Version:", scipy.__version__)
print("Pandas Version:", pandas.__version__)

In [2]:
!nvidia-smi

In [3]:
import tensorflow as tf
print("TF Version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices('GPU'))

In [4]:
Step 2: Set Environment Paths for CUDA 11.8
import os
os.environ['CUDA_HOME'] = '/usr/local/cuda-11.8'
os.environ['PATH'] += ':/usr/local/cuda-11.8/bin'
os.environ['LD_LIBRARY_PATH'] += ':/usr/local/cuda-11.8/lib64'

In [5]:

try:
    df = cudf.DataFrame({'col1': [1, 2, 3], 'col2': [4, 5, 6]})
    print("cuDF is working and using GPU!")
except Exception as e:
    print(f" cuDF GPU check failed: {e}")



In [6]:
import os
import time
import gc
import numpy as np
import pandas as pd
import xgboost as xgb
import yfinance as yf
import gymnasium as gym  Use gymnasium instead of gym
import gym_anytrading
from gymnasium.envs.registration import registry, register
import matplotlib.pyplot as plt

Prevent cuDF from taking all GPU memory
os.environ["RAPIDS_NO_INITIALIZE"] = "1"

RAPIDS & GPU-based Libraries (Try-Except to Avoid CPU Errors)
try:
    import cudf
    import cuml
    from cuml.ensemble import RandomForestClassifier
    from cuml.metrics import accuracy_score
    GPU_AVAILABLE = True
    print("cuDF & cuML are available and running on GPU.")
except ImportError:
    print(" cuDF/cuML not available. Switching to CPU mode.")
    GPU_AVAILABLE = False

Reinforcement Learning & Trading
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

TensorFlow & GPU Optimization
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

Ensure TensorFlow GPU Memory Allocation is Configured
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)  Prevents full allocation
        print("TensorFlow GPU memory growth enabled")
    except RuntimeError as e:
        print(f" TensorFlow GPU memory issue: {e}")

Google Drive Access (for Colab)
from google.colab import drive


In [7]:
Step 6: Download Stock Data
def download_stock_data(ticker, period="720d", interval="1h", max_retries=5):
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Attempt {attempt}: Downloading {ticker} stock data...")
            df_live = yf.download(ticker, period=period, interval=interval)
            if not df_live.empty:
                print("Successfully downloaded stock data!")
                df_live.reset_index(inplace=True)
                return df_live
            raise ValueError("Downloaded data is empty. Retrying...")
        except Exception as e:
            print(f" Error: {e}. Retrying in {attempt * 5} seconds...")
            time.sleep(attempt * 5)
    print(" Failed to download stock data after multiple attempts.")
    return None

df_live = download_stock_data("TSLA")
if df_live is None:
    print(" Using previously saved dataset instead.")
    file_path = '/content/drive/My Drive/teslafeature_engineered_dataset.csv'
    df_live = pd.read_csv(file_path)

In [8]:
Choose Dataset for Training
df = df_live.copy()

In [9]:
Step 3: Fix Potential Multi-Index Issues (Optional Safety Check)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

In [10]:
Step 2: Feature Engineering
=============================
df['SMA_20'] = df['Close'].rolling(window=20).mean()
df['STD_20'] = df['Close'].rolling(window=20).std()
df['Upper_Band'] = df['SMA_20'] + 2 * df['STD_20']
df['Lower_Band'] = df['SMA_20'] - 2 * df['STD_20']
df['Lowest_Low'] = df['Low'].rolling(window=14).min()
df['Highest_High'] = df['High'].rolling(window=14).max()
df['Stoch'] = ((df['Close'] - df['Lowest_Low']) / (df['Highest_High'] - df['Lowest_Low'])) * 100
df.dropna(inplace=True)

In [11]:
Create Trade Labels
df['Future_Close'] = df['Close'].shift(-10)
df['Price_Change'] = (df['Future_Close'] - df['Close']) / df['Close']
df['Target'] = np.where(df['Price_Change'] > 0.03, 1, 0)
df.dropna(inplace=True)

In [12]:
Define Features & Target
features = ['SMA_20', 'STD_20', 'Upper_Band', 'Lower_Band', 'Stoch']
X = df[features]
y = df['Target']

Convert DataFrame to GPU (if available)
if GPU_AVAILABLE:
    try:
        X = cudf.DataFrame.from_pandas(X)  Convert features to cuDF
        y = cudf.Series(y.values)  Convert target to cuDF
    except Exception as e:
        print(f" GPU Conversion Failed: {e}. Using CPU Instead.")
        X = X  Stay in Pandas format
        y = y
        GPU_AVAILABLE = False  Fallback to CPU

Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)


In [13]:

Define Feature Columns & Target Variable
feature_columns = X_train.columns  Extract feature names from training set

Train Random Forest Model
rf_model = RandomForestClassifier(n_estimators=50, random_state=42)
rf_model.fit(X_train, y_train)

Evaluate Accuracy
rf_accuracy = accuracy_score(y_test, rf_model.predict(X_test))
print(f"Random Forest Accuracy: {rf_accuracy:.4f}")

Free Memory
del X_train, X_test, y_train, y_test
gc.collect()


/usr/local/lib/python3.11/dist-packages/cuml/internals/api_decorators.py:344: UserWarning: For reproducible results in Random Forest Classifier or for almost reproducible results in Random Forest Regressor, n_streams=1 is recommended. If n_streams is > 1, results may vary due to stream/thread timing differences, even when random_state is set
  return func(**kwargs)
/usr/local/lib/python3.11/dist-packages/cuml/internals/api_decorators.py:188: UserWarning: To use pickling first train using float32 data to fit the estimator
  ret = func(*args, **kwargs)


Random Forest Accuracy: 0.7037


In [18]:
Ensure df Contains Required Features
missing_features = [col for col in feature_columns if col not in df.columns]
if missing_features:
    raise ValueError(f" Missing features in df: {missing_features}")

Generate Trade Signals (BUY = 1, SELL = 0)
df['Trade_Signal_RF'] = rf_model.predict(df[feature_columns])


In [19]:
Generate Trade Predictions (Avoid Lookahead Bias)
df['Trade_Signal_RF'] = rf_model.predict(df[feature_columns].shift(1))  Shift by 1 to prevent lookahead bias

Initialize Portfolio Metrics
portfolio_values_rf = []
capital_rf = 100000  Initial balance
shares_rf = 0
buy_price_rf = None
max_portfolio_value_rf = capital_rf  Track peak value for max drawdown calculation

Define Trade Size (Percentage of Capital Used per Trade)
trade_size_rf = 0.05  Trade 5% of capital per trade

Iterate Through Each Trade Signal
for i, trade in enumerate(df['Trade_Signal_RF']):
    price = df['Close'].iloc[i]

    BUY Signal: Invest only a fraction of capital
    if trade == 1 and capital_rf >= price and buy_price_rf is None:
        trade_amount = capital_rf * trade_size_rf  Only invest a fraction
        shares_rf = trade_amount // price

        Ensure capital doesn't get negative before executing trade
        if capital_rf - (shares_rf * price) < 0:
            print(f" Error: Overtrading detected at Step {i}, Capital=${capital_rf:,.2f}, Shares={shares_rf}")
            continue  Skip the trade to prevent negative balance

        buy_price_rf = price
        capital_rf -= shares_rf * price  Execute Buy Trade

    SELL Signal: Sell only if shares are held
    elif trade == 0 and shares_rf > 0:
        capital_rf += shares_rf * price
        shares_rf = 0
        buy_price_rf = None

    Track Portfolio Value
    portfolio_value_rf = capital_rf + (shares_rf * price)
    portfolio_values_rf.append(portfolio_value_rf)

    Track Max Drawdown
    max_portfolio_value_rf = max(max_portfolio_value_rf, portfolio_value_rf)

 Debugging: Detect Unrealistic Growth
    if shares_rf == 0 and capital_rf > 150000:  Allow reasonable growth
        print(f" Unrealistic Growth Detected at Step {i}: Capital=${capital_rf:,.2f}, Shares={shares_rf}")

Convert Portfolio Values to DataFrame
results_df_rf = pd.DataFrame({'Date': df.index, 'Portfolio Value': portfolio_values_rf})

Compute Daily Returns
results_df_rf['Daily Return'] = results_df_rf['Portfolio Value'].pct_change().fillna(0)

Compute Performance Metrics
rf_cumulative_return = ((results_df_rf['Portfolio Value'].iloc[-1] / 100000) - 1) * 100
daily_return_mean_rf = results_df_rf['Daily Return'].mean()
daily_return_std_rf = results_df_rf['Daily Return'].std()
rf_sharpe_ratio = (daily_return_mean_rf / daily_return_std_rf) * np.sqrt(252) if daily_return_std_rf != 0 else 0
drawdown_rf = (results_df_rf['Portfolio Value'].cummax() - results_df_rf['Portfolio Value']) / results_df_rf['Portfolio Value'].cummax()
rf_max_drawdown = drawdown_rf.max() * 100  Convert to percentage

Print Final Random Forest Performance
print("\n**Random Forest Portfolio Performance**")
print(f"Random Forest Accuracy: {rf_accuracy:.4f}")
print(f"RF Final Portfolio Value: ${results_df_rf['Portfolio Value'].iloc[-1]:,.2f}")
print(f"RF Cumulative Return: {rf_cumulative_return:.2f}%")
print(f"RF Sharpe Ratio: {rf_sharpe_ratio:.2f}")
print(f"RF Max Drawdown: {rf_max_drawdown:.2f}%")

Free Memory
gc.collect()



**Random Forest Portfolio Performance**
Random Forest Accuracy: 0.7037
RF Final Portfolio Value: $125,415.29
RF Cumulative Return: 25.42%
RF Sharpe Ratio: 2.11
RF Max Drawdown: 0.48%


In [20]:
import numpy as np
import pandas as pd
import gc
import xgboost as xgb
from sklearn.metrics import accuracy_score

Choose Dataset for Training
df = df_live.copy()

Fix MultiIndex Issues (if applicable)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

Remove Named Index (Fixes KeyError Issues)
df.columns.name = None  Remove 'Price' as the column index name

Ensure EMA_10 and EMA_50 exist before XGBoost training
if 'EMA_10' not in df.columns or 'EMA_50' not in df.columns:
    print(" 'EMA_10' or 'EMA_50' missing. Recomputing EMAs...")
    df['EMA_10'] = df['Close'].ewm(span=10, adjust=False).mean()
    df['EMA_50'] = df['Close'].ewm(span=50, adjust=False).mean()
    df.dropna(subset=['EMA_10', 'EMA_50'], inplace=True)  Drop NaNs from EMA calculations
    print("EMAs computed successfully.")

Generate Trade Signals (BUY = 1, SELL = 0)
df['Trade_Signal'] = 0  Default to SELL (0)

Identify BUY signals (when EMA_10 crosses above EMA_50)
df.loc[df['EMA_10'] > df['EMA_50'], 'Trade_Signal'] = 1

Ensure Trade Signal Exists
if 'Trade_Signal' not in df.columns or df['Trade_Signal'].isnull().all():
    raise ValueError(" Trade_Signal column is still missing or empty. Check feature calculations!")

print("Trade signals generated successfully!")

Define Feature Columns (Exclude Target Column)
feature_columns = ['Close', 'EMA_10', 'EMA_50']
target_column = 'Trade_Signal'

Drop NaN Values (if any)
df = df.dropna(subset=feature_columns + [target_column])

Split Data into Train & Test Sets
train_size = int(0.8 * len(df))  80% Training, 20% Testing
X_train_xgb, y_train_xgb = df[feature_columns][:train_size], df[target_column][:train_size]
X_test_xgb, y_test_xgb = df[feature_columns][train_size:], df[target_column][train_size:]

Train XGBoost Model
GPU_AVAILABLE = True  Set to False if no GPU is available
xgb_model = xgb.XGBClassifier(
    n_estimators=50,
    learning_rate=0.1,
    tree_method='hist' if GPU_AVAILABLE else 'exact',
    random_state=42
)
xgb_model.fit(X_train_xgb, y_train_xgb)

Compute Accuracy
xgb_accuracy = accuracy_score(y_test_xgb, xgb_model.predict(X_test_xgb))
print(f"XGBoost Accuracy: {xgb_accuracy:.4f}")

Free Memory
del X_train_xgb, y_train_xgb, X_test_xgb, y_test_xgb
gc.collect()

 'EMA_10' or 'EMA_50' missing. Recomputing EMAs...
EMAs computed successfully.
Trade signals generated successfully!
XGBoost Accuracy: 0.6836


In [21]:
Generate Trade Predictions for Portfolio Simulation
df['Trade_Signal_XGB'] = xgb_model.predict(df[feature_columns])

Portfolio Simulation for XGBoost Model
portfolio_values_xgb = []
capital_xgb = 100000
shares_xgb = 0
buy_price_xgb = None
max_portfolio_value_xgb = capital_xgb

for i, trade in enumerate(df['Trade_Signal_XGB']):
    price = df['Close'].iloc[i]

    if trade == 1 and capital_xgb >= price and buy_price_xgb is None:
        shares_xgb = capital_xgb // price
        buy_price_xgb = price
        capital_xgb -= shares_xgb * price
    elif trade == 0 and shares_xgb > 0:
        capital_xgb += shares_xgb * price
        shares_xgb = 0
        buy_price_xgb = None

    Update Portfolio Value
    portfolio_value_xgb = capital_xgb + (shares_xgb * price)
    portfolio_values_xgb.append(portfolio_value_xgb)

    Track Max Drawdown
    max_portfolio_value_xgb = max(max_portfolio_value_xgb, portfolio_value_xgb)

Convert Portfolio Values to DataFrame
results_df_xgb = pd.DataFrame({'Date': df.index, 'Portfolio Value': portfolio_values_xgb})

Compute Performance Metrics for XGBoost
results_df_xgb['Daily Return'] = results_df_xgb['Portfolio Value'].pct_change().fillna(0)
xgb_cumulative_return = ((results_df_xgb['Portfolio Value'].iloc[-1] / 100000) - 1) * 100
daily_return_mean_xgb = results_df_xgb['Daily Return'].mean()
daily_return_std_xgb = results_df_xgb['Daily Return'].std()
xgb_sharpe_ratio = (daily_return_mean_xgb / daily_return_std_xgb) * np.sqrt(252) if daily_return_std_xgb != 0 else 0
drawdown_xgb = (results_df_xgb['Portfolio Value'].cummax() - results_df_xgb['Portfolio Value']) / results_df_xgb['Portfolio Value'].cummax()
xgb_max_drawdown = drawdown_xgb.max() * 100

Print Performance Metrics
print("\n**XGBoost Model Performance**")
print(f"XGBoost Final Portfolio Value: ${results_df_xgb['Portfolio Value'].iloc[-1]:,.2f}")
print(f"XGBoost Cumulative Return: {xgb_cumulative_return:.2f}%")
print(f"XGBoost Sharpe Ratio: {xgb_sharpe_ratio:.2f}")
print(f"XGBoost Max Drawdown: {xgb_max_drawdown:.2f}%")


**XGBoost Model Performance**
XGBoost Final Portfolio Value: $240,328.61
XGBoost Cumulative Return: 140.33%
XGBoost Sharpe Ratio: 0.37
XGBoost Max Drawdown: 32.03%


In [22]:
import numpy as np
import pandas as pd
import gc
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

Choose Dataset for Training
df = df_live.copy()

Fix MultiIndex Issues (if applicable)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

Remove Named Index (Fixes KeyError Issues)
df.columns.name = None

Ensure EMA_10 and EMA_50 exist before GBM training
if 'EMA_10' not in df.columns or 'EMA_50' not in df.columns:
    print(" 'EMA_10' or 'EMA_50' missing. Recomputing EMAs...")
    df['EMA_10'] = df['Close'].ewm(span=10, adjust=False).mean()
    df['EMA_50'] = df['Close'].ewm(span=50, adjust=False).mean()
    df.dropna(subset=['EMA_10', 'EMA_50'], inplace=True)
    print("EMAs computed successfully.")

Generate Trade Signals (BUY = 1, SELL = 0)
df['Trade_Signal'] = 0
df.loc[df['EMA_10'] > df['EMA_50'], 'Trade_Signal'] = 1

Ensure Trade Signal Exists
if 'Trade_Signal' not in df.columns or df['Trade_Signal'].isnull().all():
    raise ValueError(" Trade_Signal column is still missing or empty. Check feature calculations!")

print("Trade signals generated successfully!")

Define Features and Target
feature_columns = ['Close', 'EMA_10', 'EMA_50']
target_column = 'Trade_Signal'

Drop NaN Values
df = df.dropna(subset=feature_columns + [target_column])

Split Data into Train & Test Sets
train_size = int(0.8 * len(df))
X_train_gb, y_train_gb = df[feature_columns][:train_size], df[target_column][:train_size]
X_test_gb, y_test_gb = df[feature_columns][train_size:], df[target_column][train_size:]

Scale Features (Optional: Helps with Some Models)
scaler = StandardScaler()
X_train_gb = pd.DataFrame(scaler.fit_transform(X_train_gb), columns=feature_columns)
X_test_gb = pd.DataFrame(scaler.transform(X_test_gb), columns=feature_columns)

print(f"Training Data Shape: {X_train_gb.shape}, {y_train_gb.shape}")
print(f"Test Data Shape: {X_test_gb.shape}, {y_test_gb.shape}")

Train Gradient Boosting Model
gb_model = GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, random_state=42)
gb_model.fit(X_train_gb, y_train_gb)

Compute Accuracy
gb_accuracy = accuracy_score(y_test_gb, gb_model.predict(X_test_gb))
print(f"Gradient Boosting Accuracy: {gb_accuracy:.4f}")

Free Memory
del X_train_gb, X_test_gb, y_train_gb, y_test_gb
gc.collect()

 'EMA_10' or 'EMA_50' missing. Recomputing EMAs...
EMAs computed successfully.
Trade signals generated successfully!
Training Data Shape: (4008, 3), (4008,)
Test Data Shape: (1002, 3), (1002,)
Gradient Boosting Accuracy: 0.6537


In [23]:
df['Trade_Signal_GB'] = gb_model.predict(df[feature_columns])

Portfolio Simulation for Gradient Boosting Model
portfolio_values_gb = []
capital_gb = 100000
shares_gb = 0
buy_price_gb = None
max_portfolio_value_gb = capital_gb

for i, trade in enumerate(df['Trade_Signal_GB']):
    price = df['Close'].iloc[i]

    if trade == 1 and capital_gb >= price and buy_price_gb is None:
        shares_gb = capital_gb // price
        buy_price_gb = price
        capital_gb -= shares_gb * price
    elif trade == 0 and shares_gb > 0:
        capital_gb += shares_gb * price
        shares_gb = 0
        buy_price_gb = None

    Update Portfolio Value
    portfolio_value_gb = capital_gb + (shares_gb * price)
    portfolio_values_gb.append(portfolio_value_gb)

    Track Max Drawdown
    max_portfolio_value_gb = max(max_portfolio_value_gb, portfolio_value_gb)

Convert Portfolio Values to DataFrame
results_df_gb = pd.DataFrame({'Date': df.index, 'Portfolio Value': portfolio_values_gb})

Compute Performance Metrics for Gradient Boosting
results_df_gb['Daily Return'] = results_df_gb['Portfolio Value'].pct_change().fillna(0)
gb_cumulative_return = ((results_df_gb['Portfolio Value'].iloc[-1] / 100000) - 1) * 100
daily_return_mean_gb = results_df_gb['Daily Return'].mean()
daily_return_std_gb = results_df_gb['Daily Return'].std()
gb_sharpe_ratio = (daily_return_mean_gb / daily_return_std_gb) * np.sqrt(252) if daily_return_std_gb != 0 else 0
drawdown_gb = (results_df_gb['Portfolio Value'].cummax() - results_df_gb['Portfolio Value']) / results_df_gb['Portfolio Value'].cummax()
gb_max_drawdown = drawdown_gb.max() * 100

Print Performance Metrics
print("\n**Gradient Boosting Model Performance**")
print(f"Gradient Boosting Final Portfolio Value: ${results_df_gb['Portfolio Value'].iloc[-1]:,.2f}")
print(f"Gradient Boosting Cumulative Return: {gb_cumulative_return:.2f}%")
print(f"Gradient Boosting Sharpe Ratio: {gb_sharpe_ratio:.2f}")
print(f"Gradient Boosting Max Drawdown: {gb_max_drawdown:.2f}%")


**Gradient Boosting Model Performance**
Gradient Boosting Final Portfolio Value: $84,693.72
Gradient Boosting Cumulative Return: -15.31%
Gradient Boosting Sharpe Ratio: 0.07
Gradient Boosting Max Drawdown: 70.07%


In [24]:
Ensure Column Names Are Flattened Correctly
df.columns = df.columns.get_level_values(0) if isinstance(df.columns, pd.MultiIndex) else df.columns

Remove Named Index (if exists)
df.columns.name = None  Remove 'Price' as the column index name

Verify Columns After Processing
print("Final Columns in df:", df.columns)

Ensure 'Close' Exists Before Proceeding
if 'Close' not in df.columns:
    raise KeyError(" Column 'Close' not found in the DataFrame after processing.")

Load Your Dataset
data = df.copy()  Ensure 'df' is loaded before this step

Add Technical Indicators (RSI & MACD)
def compute_rsi(data, window=14):
    delta = data['Close'].diff(1)
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))  Compute RSI
    return rsi  Return computed RSI values

def compute_macd(data, short=12, long=26, signal=9):
    short_ema = data['Close'].ewm(span=short, adjust=False).mean()
    long_ema = data['Close'].ewm(span=long, adjust=False).mean()
    data['MACD'] = short_ema - long_ema
    data['MACD_Signal'] = data['MACD'].ewm(span=signal, adjust=False).mean()

Apply Indicators to `data`
data['RSI'] = compute_rsi(data)  Ensure RSI is assigned properly
compute_macd(data)  Compute MACD in place

Drop NaN values to avoid errors in training
data.dropna(inplace=True)

Verify if 'RSI' column exists
print(data.head())  Print first few rows to check if RSI is present

Drop non-numeric columns before normalization (keep index intact)
data_numeric = data.select_dtypes(include=[np.number])  Keep only numeric columns

Normalize only numeric data
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data_numeric)

Convert back to DataFrame and retain the original index
scaled_df = pd.DataFrame(scaled_data, columns=data_numeric.columns, index=data.index)

Check if the data looks correct
print(scaled_df.head())


Final Columns in df: Index(['Datetime', 'Close', 'High', 'Low', 'Open', 'Volume', 'EMA_10',
       'EMA_50', 'Trade_Signal', 'Trade_Signal_GB'],
      dtype='object')


In [25]:
import os
import time
import pandas as pd
import torch
import gymnasium as gym
import gym_anytrading
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
import gc

Ensure GPU Availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Load Dataset
df = df_live.copy()

Fix MultiIndex Issues (if applicable)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

Remove Named Index (Fixes KeyError Issues)
df.columns.name = None

Ensure EMA Features Exist
if 'EMA_10' not in df.columns or 'EMA_50' not in df.columns:
    print(" 'EMA_10' or 'EMA_50' missing. Recomputing EMAs...")
    df['EMA_10'] = df['Close'].ewm(span=10, adjust=False).mean()
    df['EMA_50'] = df['Close'].ewm(span=50, adjust=False).mean()
    df.dropna(subset=['EMA_10', 'EMA_50'], inplace=True)
    print("EMAs computed successfully.")

Verify Columns Before PPO Training
print("  Final Columns Before PPO Training:", df.columns)

Split Data Into Training & Testing
train_size = int(0.8 * len(df))
df_train = df.iloc[:train_size]
df_test = df.iloc[train_size:]

Create Training Environment
env_train = gym.make('stocks-v0', df=df_train, frame_bound=(10, len(df_train)), window_size=10)
env_train = DummyVecEnv([lambda: env_train])

Train PPO Model
ppo_model = PPO(
    "MlpPolicy",
    env_train,
    verbose=1,
    learning_rate=0.0001,
    batch_size=512,
    gamma=0.995,
    n_steps=32768,
    ent_coef=0.005,
    vf_coef=0.8,
    clip_range=0.2,
    device=device,
)

ppo_model.learn(total_timesteps=500000)
ppo_model.save("ppo_trading_model_v4")
gc.collect()

Create Test Environment
env_test = gym.make('stocks-v0', df=df_test, frame_bound=(10, len(df_test)), window_size=10)
env_test = DummyVecEnv([lambda: env_test])
obs = env_test.reset()

PPO Backtesting with EMA Filtering
trade_log_rl = []
ppo_portfolio_values = []
balance_ppo = 100000
position = 0
buy_price = None

for i in range(len(df_test)):
    action, _ = ppo_model.predict(obs)
    price = df_test['Close'].iloc[i]

    Convert PPO output to discrete BUY/SELL actions
    if action < -0.3 and buy_price is not None:
        trade_log_rl.append("SELL")
        balance_ppo = position * price
        position = 0
        buy_price = None
    elif action > 0.3 and buy_price is None:
        trade_log_rl.append("BUY")
        position = balance_ppo / price
        balance_ppo = 0
        buy_price = price
    else:
        trade_log_rl.append("HOLD")

    ppo_portfolio_values.append(balance_ppo if balance_ppo > 0 else position * price)

Convert Portfolio Values to DataFrame
results_df_ppo = pd.DataFrame({'Date': df_test.index, 'Portfolio Value': ppo_portfolio_values})

Compute Daily Returns
results_df_ppo['Daily Return'] = results_df_ppo['Portfolio Value'].pct_change().fillna(0)

Compute Performance Metrics
ppo_cumulative_return = ((results_df_ppo['Portfolio Value'].iloc[-1] / 100000) - 1) * 100
daily_return_mean_ppo = results_df_ppo['Daily Return'].mean()
daily_return_std_ppo = results_df_ppo['Daily Return'].std()
ppo_sharpe_ratio = (daily_return_mean_ppo / daily_return_std_ppo) * np.sqrt(252) if daily_return_std_ppo != 0 else 0
drawdown_ppo = (results_df_ppo['Portfolio Value'].cummax() - results_df_ppo['Portfolio Value']) / results_df_ppo['Portfolio Value'].cummax()
ppo_max_drawdown = drawdown_ppo.max() * 100

Buy & Hold Strategy for Comparison
initial_balance = 100000
shares_held = initial_balance // df_test['Close'].iloc[0]  Ensure whole number shares
buy_hold_final_value = shares_held * df_test['Close'].iloc[-1]
buy_hold_cumulative_return = ((buy_hold_final_value / initial_balance) - 1) * 100

Print Final Performance
print("\nFINAL RESULTS COMPARISON:")
print("\n**Reinforcement Learning (PPO)**")
print(f"PPO Final Portfolio Value: ${results_df_ppo['Portfolio Value'].iloc[-1]:,.2f}")
print(f"PPO Cumulative Return: {ppo_cumulative_return:.2f}%")
print(f"PPO Sharpe Ratio: {ppo_sharpe_ratio:.2f}")
print(f"PPO Max Drawdown: {ppo_max_drawdown:.2f}%")

print("\n**Buy & Hold Baseline**")
print(f"Buy & Hold Final Portfolio Value: ${buy_hold_final_value:,.2f}")
print(f"Buy & Hold Cumulative Return: {buy_hold_cumulative_return:.2f}%")

Declare Winner Based on Final Portfolio Value
best_strategy = "PPO" if results_df_ppo['Portfolio Value'].iloc[-1] > buy_hold_final_value else "Buy & Hold"
print(f"\n**Best Strategy Based on Final Portfolio Value: {best_strategy}!**")

Free Memory
gc.collect()


Using device: cuda
 'EMA_10' or 'EMA_50' missing. Recomputing EMAs...
EMAs computed successfully.
  Final Columns Before PPO Training: Index(['Datetime', 'Close', 'High', 'Low', 'Open', 'Volume', 'EMA_10',
       'EMA_50'],
      dtype='object')
Using cuda device


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


FINAL RESULTS COMPARISON:

**Reinforcement Learning (PPO)**
PPO Final Portfolio Value: $124,986.77
PPO Cumulative Return: 24.99%
PPO Sharpe Ratio: 0.35
PPO Max Drawdown: 41.29%

**Buy & Hold Baseline**
Buy & Hold Final Portfolio Value: $121,892.10
Buy & Hold Cumulative Return: 21.89%

**Best Strategy Based on Final Portfolio Value: PPO!**


In [26]:
import subprocess

packages = [
    "stable-baselines3[extra]",
    "gymnasium",
    "gym-anytrading",
    "numpy",
    "pandas",
    "matplotlib"
]

for pkg in packages:
    subprocess.run(["pip", "install", "--upgrade", "--force-reinstall", pkg])


In [27]:
import numpy as np
import pandas as pd
import gc
import gymnasium as gym
import gym_anytrading
import torch
from stable_baselines3.common.vec_env import DummyVecEnv

Load & Preprocess Data
df = df_live.copy()

Fix MultiIndex Issues (if applicable)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

Remove Named Index (Fixes KeyError Issues)
df.columns.name = None

Ensure Essential Columns Exist
required_cols = {'Close', 'High', 'Low', 'Open', 'Volume'}
missing_cols = required_cols - set(df.columns)
if missing_cols:
    raise ValueError(f" Missing required columns: {missing_cols}")

Compute EMAs if missing
if 'EMA_10' not in df.columns or 'EMA_50' not in df.columns:
    print(" 'EMA_10' or 'EMA_50' missing. Recomputing EMAs...")
    df['EMA_10'] = df['Close'].ewm(span=10, adjust=False).mean()
    df['EMA_50'] = df['Close'].ewm(span=50, adjust=False).mean()
    df.dropna(subset=['EMA_10', 'EMA_50'], inplace=True)
    print("EMAs computed successfully.")

Confirm EMA columns before Training
if 'EMA_10' not in df.columns or 'EMA_50' not in df.columns:
    raise KeyError(" EMA columns are still missing after recomputation!")

Final Data Check
print("  Final Columns Before Training:", df.columns)
df.reset_index(drop=True, inplace=True)

Enable GPU Acceleration (if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Create Trading Environment
env = gym.make('stocks-v0', df=df, frame_bound=(10, len(df) - 1), window_size=10)
env = DummyVecEnv([lambda: env])

Define SARSA Agent with Portfolio Tracking
class SARSAAgent:
    def __init__(self, state_size, action_size, alpha=0.1, gamma=0.95, epsilon=1.0, epsilon_decay=0.995, min_epsilon=0.01):
        self.state_size = state_size
        self.action_size = action_size
        self.q_table = np.zeros((state_size, action_size))
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.min_epsilon = min_epsilon
        self.portfolio_values = []  Track Portfolio Values

    def choose_action(self, state):
        if np.random.rand() < self.epsilon:
            return np.array([np.random.choice(self.action_size)])  Return as array (Fix)
        return np.array([np.argmax(self.q_table[state])])  Return as array (Fix)

    def update_q_table(self, state, action, reward, next_state, next_action):
        predict = self.q_table[state, action[0]]  Extract int from action array
        target = reward + self.gamma * self.q_table[next_state, next_action[0]]
        self.q_table[state, action[0]] += self.alpha * (target - predict)

    def train(self, env, episodes=500, initial_balance=10000):
        for episode in range(episodes):
            reset_result = env.reset()
            state = reset_result[0] if isinstance(reset_result, tuple) else reset_result  Fix reset return issue
            state = self.discretize_state(state)
            action = self.choose_action(state)
            done = False
            total_reward = 0
            balance = initial_balance  Initialize Portfolio Balance

            while not done:
                step_result = env.step(action)
                if isinstance(step_result, tuple) and len(step_result) == 4:  Fix Unpacking Issue
                    next_state, reward, done, _ = step_result
                else:
                    next_state, reward, done = step_result

                next_state = self.discretize_state(next_state)
                next_action = self.choose_action(next_state)
                self.update_q_table(state, action, reward, next_state, next_action)

                Update Portfolio Value (Simulating Balance)
                balance += reward
                self.portfolio_values.append(balance)  Store Portfolio Value

                state, action = next_state, next_action
                total_reward += reward

            self.epsilon = max(self.epsilon * self.epsilon_decay, self.min_epsilon)
            print(f"Episode {episode + 1}, Total Reward: {total_reward}, Final Balance: {balance}")

    def discretize_state(self, state):
        """Convert continuous states into discrete bins for Q-learning."""
        return int(state.mean() * self.state_size) % self.state_size

Initialize and Train SARSA
state_size = 100
action_size = env.action_space.n
sarsa_agent = SARSAAgent(state_size, action_size)
sarsa_agent.train(env, episodes=500)  Now it will store portfolio values!

 'EMA_10' or 'EMA_50' missing. Recomputing EMAs...
EMAs computed successfully.
  Final Columns Before Training: Index(['Datetime', 'Close', 'High', 'Low', 'Open', 'Volume', 'EMA_10',
       'EMA_50'],
      dtype='object')
Using device: cuda


<ipython-input-27-aa44d7d3f5df>:70: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  self.q_table[state, action[0]] += self.alpha * (target - predict)


In [28]:
import numpy as np
import pandas as pd
import gc

Ensure SARSA Stores Portfolio Values
if not hasattr(sarsa_agent, 'portfolio_values') or not sarsa_agent.portfolio_values:
    raise ValueError(" Error: `sarsa_portfolio_values` is missing or empty. Check if SARSA training stored portfolio values correctly.")

Convert SARSA Portfolio Values to NumPy Array & Flatten
sarsa_portfolio_values = np.array([val.squeeze() for val in sarsa_agent.portfolio_values], dtype=float)

Ensure DataFrame and Portfolio Length Match
min_length = min(len(df), len(sarsa_portfolio_values))
df = df.iloc[:min_length]  Trim DataFrame to match portfolio values length
sarsa_portfolio_values = sarsa_portfolio_values[:min_length]  Trim SARSA portfolio values

Debugging Print
print(f"Data Length: {len(df)}, SARSA Portfolio Length: {len(sarsa_portfolio_values)}")

Convert SARSA Portfolio Values to DataFrame
results_df_sarsa = pd.DataFrame({
    'Date': df.index[:min_length],
    'Portfolio Value': sarsa_portfolio_values
})

Compute Daily Returns (Fix Array Issue)
results_df_sarsa['Daily Return'] = results_df_sarsa['Portfolio Value'].pct_change().fillna(0)

Compute SARSA Performance Metrics
if len(results_df_sarsa) < 2:
    raise ValueError(" Error: Not enough data points to compute cumulative return.")

sarsa_cumulative_return = ((results_df_sarsa['Portfolio Value'].iloc[-1] / results_df_sarsa['Portfolio Value'].iloc[0]) - 1) * 100
daily_return_mean_sarsa = results_df_sarsa['Daily Return'].mean()
daily_return_std_sarsa = results_df_sarsa['Daily Return'].std()
sarsa_sharpe_ratio = (daily_return_mean_sarsa / daily_return_std_sarsa) * np.sqrt(252) if daily_return_std_sarsa != 0 else 0
drawdown_sarsa = (results_df_sarsa['Portfolio Value'].cummax() - results_df_sarsa['Portfolio Value']) / results_df_sarsa['Portfolio Value'].cummax()
sarsa_max_drawdown = drawdown_sarsa.max() * 100

Print Final Results
print("\n**Reinforcement Learning (SARSA)**")
print(f"SARSA Final Portfolio Value: ${sarsa_portfolio_values[-1]:,.2f}")
print(f"SARSA Cumulative Return: {sarsa_cumulative_return:.2f}%")
print(f"SARSA Sharpe Ratio: {sarsa_sharpe_ratio:.2f}")
print(f"SARSA Max Drawdown: {sarsa_max_drawdown:.2f}%")

Free Memory
gc.collect()


Data Length: 5010, SARSA Portfolio Length: 5010

**Reinforcement Learning (SARSA)**
SARSA Final Portfolio Value: $10,216.28
SARSA Cumulative Return: 1.75%
SARSA Sharpe Ratio: 0.22
SARSA Max Drawdown: 0.00%


In [29]:
import numpy as np
import pandas as pd
import torch
import gymnasium as gym
import gym_anytrading
from stable_baselines3 import DDPG
from stable_baselines3.common.noise import NormalActionNoise
from stable_baselines3.common.vec_env import DummyVecEnv
from gymnasium.spaces import Box
from gym_anytrading.envs import StocksEnv

Choose Dataset for Training
df = df_live.copy()

Fix MultiIndex Issues (if applicable)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

Remove Named Index (Fixes KeyError Issues)
df.columns.name = None

Ensure EMA_10 and EMA_50 exist before DDPG training
if 'EMA_10' not in df.columns or 'EMA_50' not in df.columns:
    print(" 'EMA_10' or 'EMA_50' missing. Recomputing EMAs...")
    df['EMA_10'] = df['Close'].ewm(span=10, adjust=False).mean()
    df['EMA_50'] = df['Close'].ewm(span=50, adjust=False).mean()
    df.dropna(subset=['EMA_10', 'EMA_50'], inplace=True)
    print("EMAs computed successfully.")

Confirm EMA columns before DDPG Training
if 'EMA_10' not in df.columns or 'EMA_50' not in df.columns:
    raise KeyError(" EMA columns are still missing after recomputation!")

print("  Final Columns Before Training:", df.columns)

Enable GPU Acceleration for DDPG (if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Modify the Trading Environment for Continuous Actions
class ContinuousTradingEnv(StocksEnv):
    def __init__(self, df, frame_bound, window_size):
        super().__init__(df=df, frame_bound=frame_bound, window_size=window_size)
        self.action_space = Box(low=-1.0, high=1.0, shape=(1,), dtype=np.float32)

    def step(self, action):
        Convert continuous action (-1 to 1) to a discrete action (BUY, SELL, HOLD)
        if action < -0.3:
            discrete_action = 0  SELL
        elif action > 0.3:
            discrete_action = 1  BUY
        else:
            discrete_action = 2  HOLD

        return super().step(discrete_action)

Initialize the Custom Environment
env = ContinuousTradingEnv(df=df, frame_bound=(10, len(df)), window_size=10)
env = DummyVecEnv([lambda: env])

Define Action Noise for DDPG
n_actions = env.action_space.shape[-1]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))

Train DDPG Model with Optimized Settings
ddpg_model = DDPG(
    "MlpPolicy",
    env,
    action_noise=action_noise,
    verbose=1,
    learning_rate=0.0001,
    batch_size=128, Reduced batch size for faster training
    gamma=0.99,
    tau=0.01, Faster target updates
    gradient_steps=2, More updates per step
    tensorboard_log="./ddpg_tensorboard/", Track progress
    device=device,  Use GPU if available
)

Train with Reduced Timesteps (100,000 instead of 500,000)
ddpg_model.learn(total_timesteps=100000)
ddpg_model.save("ddpg_trading_model_v1")

Run DDPG Trading Strategy
obs = env.reset()
trade_log_ddpg = []
buy_price = None

for i in range(len(df)):
    if i == 0:
        trade_log_ddpg.append("HOLD")
        continue

    action, _ = ddpg_model.predict(obs)

    if action < -0.3 and buy_price is not None:
        trade_log_ddpg.append("SELL")
        buy_price = None
    elif action > 0.3 and buy_price is None:
        trade_log_ddpg.append("BUY")
        buy_price = df['Close'].iloc[i]
    else:
        trade_log_ddpg.append("HOLD")

df["DDPG_Trade_Signal"] = trade_log_ddpg

Run Backtesting
initial_balance = 100000
shares_held = initial_balance / df['Close'].iloc[0]
final_balance_hold = shares_held * df['Close'].iloc[-1]

balance_ddpg = 100000
position = 0
portfolio_values_ddpg = []

for i, trade in enumerate(trade_log_ddpg):
    price = df['Close'].iloc[i]

    if trade == "BUY" and position == 0:
        position = balance_ddpg / price
        balance_ddpg = 0
    elif trade == "SELL" and position > 0:
        balance_ddpg = position * price
        position = 0

    portfolio_values_ddpg.append(balance_ddpg if balance_ddpg > 0 else position * price)

final_balance_ddpg = portfolio_values_ddpg[-1]


 'EMA_10' or 'EMA_50' missing. Recomputing EMAs...
EMAs computed successfully.
  Final Columns Before Training: Index(['Datetime', 'Close', 'High', 'Low', 'Open', 'Volume', 'EMA_10',
       'EMA_50'],
      dtype='object')
Using device: cuda
Using cuda device


In [30]:
Compute Performance Metrics for DDPG
results_df_ddpg = pd.DataFrame({'Date': df.index, 'Portfolio Value': portfolio_values_ddpg})
results_df_ddpg['Daily Return'] = results_df_ddpg['Portfolio Value'].pct_change().fillna(0)

ddpg_cumulative_return = ((results_df_ddpg['Portfolio Value'].iloc[-1] / initial_balance) - 1) * 100
ddpg_sharpe_ratio = (results_df_ddpg['Daily Return'].mean() / results_df_ddpg['Daily Return'].std()) * np.sqrt(252) if results_df_ddpg['Daily Return'].std() != 0 else 0
ddpg_max_drawdown = ((results_df_ddpg['Portfolio Value'].cummax() - results_df_ddpg['Portfolio Value']) / results_df_ddpg['Portfolio Value'].cummax()).max() * 100

Final Results Summary
print("\nFINAL RESULTS COMPARISON:")

print("**Reinforcement Learning (DDPG)**")
print(f"DDPG Final Portfolio Value: ${final_balance_ddpg:,.2f}")
print(f"DDPG Cumulative Return: {ddpg_cumulative_return:.2f}%")
print(f"DDPG Sharpe Ratio: {ddpg_sharpe_ratio:.2f}")
print(f"DDPG Max Drawdown: {ddpg_max_drawdown:.2f}%")
print(f"DDPG Trade Log (First 10): {trade_log_ddpg[:10]} ...")

print("\n**Buy & Hold Baseline**")
print(f"Buy & Hold Final Portfolio Value: ${final_balance_hold:,.2f}")

Declare Winner Based on Portfolio Performance
winner = "DDPG" if final_balance_ddpg > final_balance_hold else "Buy & Hold"
print(f"\n**Best Strategy Based on Final Portfolio Value: {winner}!**")



FINAL RESULTS COMPARISON:
**Reinforcement Learning (DDPG)**
DDPG Final Portfolio Value: $85,222.56
DDPG Cumulative Return: -14.78%
DDPG Sharpe Ratio: 0.07
DDPG Max Drawdown: 70.22%
DDPG Trade Log (First 10): ['HOLD', 'BUY', 'HOLD', 'HOLD', 'HOLD', 'HOLD', 'HOLD', 'HOLD', 'HOLD', 'HOLD'] ...

**Buy & Hold Baseline**
Buy & Hold Final Portfolio Value: $84,660.03

**Best Strategy Based on Final Portfolio Value: DDPG!**


In [31]:
import gymnasium as gym
import pandas as pd
import numpy as np
import torch
from stable_baselines3 import A2C
from stable_baselines3.common.vec_env import DummyVecEnv
from gymnasium import spaces

Load dataset (Assuming df_live contains market data)
df = df_live.copy()

Ensure Data is in Proper Format
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df.columns.name = None

Compute EMA if Missing
if 'EMA_10' not in df.columns or 'EMA_50' not in df.columns:
    print(" 'EMA_10' or 'EMA_50' missing. Recomputing EMAs...")
    df['EMA_10'] = df['Close'].ewm(span=10, adjust=False).mean()
    df['EMA_50'] = df['Close'].ewm(span=50, adjust=False).mean()
    df.dropna(subset=['EMA_10', 'EMA_50'], inplace=True)
    print("EMAs computed successfully.")

if 'EMA_10' not in df.columns or 'EMA_50' not in df.columns:
    raise KeyError(" EMA columns are still missing after recomputation!")

Define a Custom Gymnasium Environment for A2C
class TradingEnv(gym.Env):
    def __init__(self, data):
        super(TradingEnv, self).__init__()

        self.data = data[['Close', 'EMA_10', 'EMA_50']].values
        self.current_step = 0

        Define Action and Observation Spaces
        self.action_space = spaces.Discrete(2)  0 = Hold, 1 = Trade
        self.observation_space = spaces.Box(
            low=-float("inf"), high=float("inf"), shape=(3,), dtype=np.float32
        )

    def reset(self, seed=None, options=None):
        self.current_step = 0
        return self.data[self.current_step], {}

    def step(self, action):
        self.current_step += 1

        if self.current_step >= len(self.data):
            done = True
            reward = 0  No more data
            obs = self.data[-1]
        else:
            done = False
            obs = self.data[self.current_step]

            Reward System: Reward correct trades
            if action == 1 and self.data[self.current_step - 1][1] < self.data[self.current_step - 1][2] and obs[1] >= obs[2]:
                reward = 1  Buy on EMA_10 crossing EMA_50 upwards
            elif action == 0 and self.data[self.current_step - 1][1] > self.data[self.current_step - 1][2] and obs[1] <= obs[2]:
                reward = 1  Hold when EMA_10 is below EMA_50
            else:
                reward = -1  Penalize incorrect trades

        return obs, reward, done, False, {}

Create and Wrap Gymnasium Environment
env = DummyVecEnv([lambda: TradingEnv(df)])

Train A2C Model
model = A2C("MlpPolicy", env, verbose=1, device="cuda" if torch.cuda.is_available() else "cpu")
model.learn(total_timesteps=10000)

Save Model
model.save("a2c_trading_model")

print("A2C Training Completed and Model Saved!")


 'EMA_10' or 'EMA_50' missing. Recomputing EMAs...
EMAs computed successfully.
Using cuda device


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run A2C on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


A2C Training Completed and Model Saved!


In [32]:
from stable_baselines3 import A2C
from stable_baselines3.common.vec_env import DummyVecEnv
from gymnasium import spaces

Load dataset
df = df_live.copy()

Ensure Data is in Proper Format
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)
df.columns.name = None  Remove unwanted index names

Compute EMA if Missing
if 'EMA_10' not in df.columns or 'EMA_50' not in df.columns:
    print(" 'EMA_10' or 'EMA_50' missing. Recomputing EMAs...")
    df['EMA_10'] = df['Close'].ewm(span=10, adjust=False).mean()
    df['EMA_50'] = df['Close'].ewm(span=50, adjust=False).mean()
    df.dropna(subset=['EMA_10', 'EMA_50'], inplace=True)
    print("EMAs computed successfully.")

Define Custom Reward Function
def reward_function(action, price, last_price):
    """Dynamic reward function based on market conditions."""
    profit = price - last_price if action == 1 else last_price - price
    return profit * 10 if profit > 0 else profit * 5  Reward for profits, penalty for losses

Define Custom Trading Environment for A2C
class TradingEnv(gym.Env):
    def __init__(self, data):
        super(TradingEnv, self).__init__()

        self.data = data[['Close', 'EMA_10', 'EMA_50']].values
        self.current_step = 0
        self.last_price = self.data[0][0]  Track last price

        Define Action and Observation Spaces
        self.action_space = spaces.Discrete(2)  0 = Hold, 1 = Trade
        self.observation_space = spaces.Box(
            low=-float("inf"), high=float("inf"), shape=(3,), dtype=np.float32
        )

    def reset(self, seed=None, options=None):
        self.current_step = 0
        self.last_price = self.data[0][0]  Reset last price
        return self.data[self.current_step], {}

    def step(self, action):
        self.current_step += 1

        if self.current_step >= len(self.data):
            done = True
            reward = 0  No more data
            obs = self.data[-1]
        else:
            done = False
            obs = self.data[self.current_step]

            Apply Reward Function
            reward = reward_function(action, obs[0], self.last_price)
            self.last_price = obs[0]  Update last price

        return obs, reward, done, False, {}

Create & Wrap Trading Environment
env = DummyVecEnv([lambda: TradingEnv(df)])

Train A2C Model with Updated Hyperparameters
model = A2C(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=0.0003,  Increased learning rate
    gamma=0.99,  Higher discount factor for long-term rewards
    n_steps=10,  More frequent updates for stability
    ent_coef=0.01,  Reduce exploration
    device="cuda" if torch.cuda.is_available() else "cpu"
)

Increase Training Timesteps
model.learn(total_timesteps=200000)  Increased from 10,000 to 200,000

Save Model
model.save("a2c_trading_model_v2")
print("A2C Model Training Completed!")

Evaluate A2C Performance
obs = env.reset()
done = False
total_reward = 0
a2c_portfolio_values = []
initial_balance = 10000  Assume starting with $10,000

while not done:
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, done, _ = env.step(action)

    Ensure reward is a scalar value (Extract from array if necessary)
    if isinstance(reward, np.ndarray):
        reward = reward.item()  Convert NumPy array to scalar

    total_reward += reward
    initial_balance += reward  Simulating portfolio value
    a2c_portfolio_values.append(initial_balance)

Convert Portfolio Values to DataFrame
min_length = min(len(df), len(a2c_portfolio_values))
df = df.iloc[:min_length]  Trim DataFrame to match portfolio values length
a2c_portfolio_values = a2c_portfolio_values[:min_length]  Trim A2C portfolio values

results_df_a2c = pd.DataFrame({'Date': df.index, 'Portfolio Value': a2c_portfolio_values})

Compute Daily Returns
results_df_a2c['Daily Return'] = results_df_a2c['Portfolio Value'].pct_change().fillna(0)

Ensure all values are numeric before calculations
results_df_a2c['Daily Return'] = pd.to_numeric(results_df_a2c['Daily Return'], errors='coerce')

Compute Cumulative Return
a2c_cumulative_return = ((results_df_a2c['Portfolio Value'].iloc[-1] / results_df_a2c['Portfolio Value'].iloc[0]) - 1) * 100

Compute Sharpe Ratio (Annualized)
daily_return_mean_a2c = results_df_a2c['Daily Return'].mean()
daily_return_std_a2c = results_df_a2c['Daily Return'].std()

Ensure standard deviation is not zero
a2c_sharpe_ratio = (daily_return_mean_a2c / daily_return_std_a2c) * np.sqrt(252) if daily_return_std_a2c != 0 else 0

Compute Max Drawdown
drawdown_a2c = (results_df_a2c['Portfolio Value'].cummax() - results_df_a2c['Portfolio Value']) / results_df_a2c['Portfolio Value'].cummax()
a2c_max_drawdown = drawdown_a2c.max() * 100  Convert to percentage

Print A2C Model Performance
print("\n**Reinforcement Learning (A2C) Performance**")
print(f"A2C Final Portfolio Value: ${results_df_a2c['Portfolio Value'].iloc[-1]:,.2f}")
print(f"A2C Cumulative Return: {a2c_cumulative_return:.2f}%")
print(f"A2C Sharpe Ratio: {a2c_sharpe_ratio:.2f}")
print(f"A2C Max Drawdown: {a2c_max_drawdown:.2f}%")

Free Memory
import gc
gc.collect()


A2C Cumulative Return: -48.86%
A2C Sharpe Ratio: -52.41
A2C Max Drawdown: 48.86%


In [33]:
import gc

Compute Buy & Hold Performance
initial_balance = 100000  Starting portfolio value
buy_price = df['Close'].iloc[0]  Buy at the first available price
sell_price = df['Close'].iloc[-1]  Sell at the last available price

shares_held = initial_balance / buy_price
buy_hold_final_value = shares_held * sell_price  Final portfolio value

Compute Buy & Hold Metrics
buy_hold_cumulative_return = ((buy_hold_final_value / initial_balance) - 1) * 100

print("\n**Buy & Hold Baseline**")
print(f"Buy & Hold Final Portfolio Value: ${buy_hold_final_value:,.2f}")
print(f"Buy & Hold Cumulative Return: {buy_hold_cumulative_return:.2f}%")

gc.collect()



**Buy & Hold Baseline**
Buy & Hold Final Portfolio Value: $84,660.03
Buy & Hold Cumulative Return: -15.34%


In [34]:
FINAL RESULTS COMPARISON
print("\nFINAL RESULTS COMPARISON:")

Reinforcement Learning Models
print("\n**Reinforcement Learning (A2C)**")
print(f"A2C Final Portfolio Value: ${results_df_a2c['Portfolio Value'].iloc[-1]:,.2f}")
print(f"A2C Cumulative Return: {a2c_cumulative_return:.2f}%")
print(f"A2C Sharpe Ratio: {a2c_sharpe_ratio:.2f}")
print(f"A2C Max Drawdown: {a2c_max_drawdown:.2f}%")

print("\n**Reinforcement Learning (PPO)**")
print(f"PPO Final Portfolio Value: ${results_df_ppo['Portfolio Value'].iloc[-1]:,.2f}")
print(f"PPO Cumulative Return: {ppo_cumulative_return:.2f}%")
print(f"PPO Sharpe Ratio: {ppo_sharpe_ratio:.2f}")
print(f"PPO Max Drawdown: {ppo_max_drawdown:.2f}%")

print("\n**Reinforcement Learning (DDPG)**")
print(f"DDPG Final Portfolio Value: ${results_df_ddpg['Portfolio Value'].iloc[-1]:,.2f}")
print(f"DDPG Cumulative Return: {ddpg_cumulative_return:.2f}%")
print(f"DDPG Sharpe Ratio: {ddpg_sharpe_ratio:.2f}")
print(f"DDPG Max Drawdown: {ddpg_max_drawdown:.2f}%")

Buy & Hold Strategy
print("\n**Buy & Hold Baseline**")
print(f"Buy & Hold Final Portfolio Value: ${buy_hold_final_value:,.2f}")
print(f"Buy & Hold Cumulative Return: {buy_hold_cumulative_return:.2f}%")

Machine Learning Models (Portfolio Performance + Accuracy)
print("\n**Machine Learning Models (Portfolio Performance)**")

Random Forest
print(f"\n**Random Forest**")
print(f"Random Forest Accuracy: {rf_accuracy:.4f}")
print(f"Random Forest Final Portfolio Value: ${results_df_rf['Portfolio Value'].iloc[-1]:,.2f}")
print(f"Random Forest Cumulative Return: {rf_cumulative_return:.2f}%")

Gradient Boosting
print(f"\n**Gradient Boosting**")
print(f"Gradient Boosting Accuracy: {gb_accuracy:.4f}")
print(f"Gradient Boosting Final Portfolio Value: ${results_df_gb['Portfolio Value'].iloc[-1]:,.2f}")
print(f"Gradient Boosting Cumulative Return: {gb_cumulative_return:.2f}%")

XGBoost
print(f"\n**XGBoost**")
print(f"XGBoost Accuracy: {xgb_accuracy:.4f}")
print(f"XGBoost Final Portfolio Value: ${results_df_xgb['Portfolio Value'].iloc[-1]:,.2f}")
print(f"XGBoost Cumulative Return: {xgb_cumulative_return:.2f}%")

Determine Best Trading Strategy (Based on Portfolio Value)
strategy_results = {
    "A2C": results_df_a2c['Portfolio Value'].iloc[-1],
    "PPO": results_df_ppo['Portfolio Value'].iloc[-1],
    "DDPG": results_df_ddpg['Portfolio Value'].iloc[-1],
    "Random Forest": results_df_rf['Portfolio Value'].iloc[-1],
    "Gradient Boosting": results_df_gb['Portfolio Value'].iloc[-1],
    "XGBoost": results_df_xgb['Portfolio Value'].iloc[-1],
    "Buy & Hold": buy_hold_final_value
}

best_strategy = max(strategy_results, key=strategy_results.get)

Print Best Performing Strategy
print(f"\n**Best Strategy Based on Final Portfolio Value: {best_strategy}!**")

Clean up memory
gc.collect()



FINAL RESULTS COMPARISON:

**Reinforcement Learning (A2C)**
A2C Final Portfolio Value: $5,113.00
A2C Cumulative Return: -48.86%
A2C Sharpe Ratio: -52.41
A2C Max Drawdown: 48.86%

**Reinforcement Learning (PPO)**
PPO Final Portfolio Value: $124,986.77
PPO Cumulative Return: 24.99%
PPO Sharpe Ratio: 0.35
PPO Max Drawdown: 41.29%

**Reinforcement Learning (DDPG)**
DDPG Final Portfolio Value: $85,222.56
DDPG Cumulative Return: -14.78%
DDPG Sharpe Ratio: 0.07
DDPG Max Drawdown: 70.22%

**Buy & Hold Baseline**
Buy & Hold Final Portfolio Value: $84,660.03
Buy & Hold Cumulative Return: -15.34%

**Machine Learning Models (Portfolio Performance)**

**Random Forest**
Random Forest Accuracy: 0.7037
Random Forest Final Portfolio Value: $125,415.29
Random Forest Cumulative Return: 25.42%

**Gradient Boosting**
Gradient Boosting Accuracy: 0.6537
Gradient Boosting Final Portfolio Value: $84,693.72
Gradient Boosting Cumulative Return: -15.31%

**XGBoost**
XGBoost Accuracy: 0.6836
XGBoost Final Portfol